# Zero-Day-Style OOD Detection

Train an IDS classifier on **known** attack classes, then build a separate **OOD scorer** (energy-based + Mahalanobis distance on the penultimate-layer embeddings) that flags inputs which don't resemble any known-class distribution.

**Threat model:** a *zero-day style unknown attack* is one that was never part of the training data - the classifier has never seen it and its labeled class is unavailable. A supervised IDS will happily classify it into one of the known classes (overconfidence). An OOD scorer adds a second stage that says *"this input is not like anything I was trained on - escalate"*.

Here `data-exfiltration` is **deliberately excluded** from training and acts as the held-out zero-day proxy at test time.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if Path.cwd().name == "zero-day-ood-detection" else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["figure.facecolor"] = "#09090b"
plt.rcParams["axes.facecolor"] = "#131316"
plt.rcParams["axes.edgecolor"] = "#27272a"
plt.rcParams["text.color"] = "#e4e4e7"
plt.rcParams["axes.labelcolor"] = "#e4e4e7"
plt.rcParams["xtick.color"] = "#e4e4e7"
plt.rcParams["ytick.color"] = "#e4e4e7"
print(f"Python: {sys.version.split()[0]}")

## 1. Dataset

5000 synthetic network-flow samples, 18 features, 5 classes. The model trains on **benign, portscan, brute-force, malware-c2** only; `data-exfiltration` appears only in the test set.

In [ ]:
from zood.data import make_splits, CLASS_NAMES, KNOWN_CLASS_NAMES, UNKNOWN_CLASS_NAME

X_train, y_train, X_test, y_test, groups = make_splits(n_samples=5000, test_frac=0.35, random_state=42)
print("train (known only):", X_train.shape, "test:", X_test.shape)
print("known classes:", KNOWN_CLASS_NAMES)
print("held-out unknown:", UNKNOWN_CLASS_NAME)
print("test group counts:", dict(zip(*np.unique(groups, return_counts=True))))

## 2. Train the IDS classifier on known classes

Small torch MLP `18 -> 64 -> 32(embedding) -> 4`. The 32-d penultimate layer is the space used for the Mahalanobis OOD fit.

In [ ]:
from sklearn.preprocessing import StandardScaler
from zood.model import IDSMLP, train_model, predict_logits, get_embeddings
from zood.data import N_FEATURES, N_KNOWN

scaler = StandardScaler().fit(X_train)
Xs_tr = scaler.transform(X_train).astype("float32")
Xs_te = scaler.transform(X_test).astype("float32")

model = IDSMLP(n_features=N_FEATURES, n_classes=N_KNOWN, seed=42)
train_model(model, Xs_tr, y_train, epochs=40, batch_size=128, lr=1e-3, seed=42, verbose=False)
print("done")

In [ ]:
from zood.evaluate import evaluate_classifier

clf = evaluate_classifier(model, Xs_te, y_test, groups)
print(f"known-class test accuracy: {clf['acc']*100:.1f}%  macro F1: {clf['f1']:.4f}")
print("per-class:", clf["per_class"])

## 3. Build the OOD scorers (manual - no pytorch-ood)

1. **Energy-based**: `score = -E(x)` with `E(x) = -T logsumexp(logits / T)` over the class logits. Unknown inputs that the classifier can't confidently place produce low energy (high OOD score).
2. **Mahalanobis**: fit a per-class Gaussian (mean + pooled covariance with shrinkage) on the known-class *embeddings*; score each input by its minimum Mahalanobis distance to the class centroids.

In [ ]:
from zood.ood import energy_ood_score, fit_mahalanobis_reference, mahalanobis_ood_scores

logits = predict_logits(model, Xs_te)
emb_tr = get_embeddings(model, Xs_tr)
emb_te = get_embeddings(model, Xs_te)

energy = energy_ood_score(logits)
means, cov_inv, cov = fit_mahalanobis_reference(emb_tr, y_train, n_classes=N_KNOWN, shrink=0.1)
maha = mahalanobis_ood_scores(emb_te, means, cov_inv)
print("energy scores:", energy.shape, "mahalanobis:", maha.shape)

## 4. Evaluate

OOD AUROC (unknown vs known), threshold at 95% recall of the unknown class, and FPR on benign traffic / known attacks at that threshold.

In [ ]:
from zood.evaluate import evaluate_ood

res = evaluate_ood(energy, maha, groups, tpr_target=0.95)
for key, r in res.items():
    print(f"{key:>12}: AUROC={r['auroc']:.4f}  threshold@{r['tpr_target']*100:.0f}%TPR={r['threshold']:.3f}  "
          f"FPR benign={r['benign_fpr']:.4f}  FPR known={r['known_fpr']:.4f}  "
          f"separation={r['separation_std']:.2f} std")

## 5. Visualise the separation

Known vs benign vs unknown score distributions, the ROC, and the embedding space.

In [ ]:
from zood.evaluate import plot_ood_histogram, plot_auroc_curve, roc_curves

fig_dir = ROOT / "results" / "figures"
plot_ood_histogram(res, groups, fig_dir / "ood_histogram.png")
plot_auroc_curve(roc_curves(res, groups), res, fig_dir / "auroc_curve.png")
plot_embeddings_scatter(emb_te, groups, fig_dir / "embeddings_scatter.png")
from PIL import Image
for name in ["ood_histogram.png", "auroc_curve.png", "embeddings_scatter.png"]:
    im = Image.open(fig_dir / name); plt.figure(); plt.imshow(im); plt.axis("off")

## 6. Summary

The energy and Mahalanobis OOD scorers both clearly separate the held-out `data-exfiltration` class (OOD AUROC > 0.87) from known traffic, while the supervised classifier still scores ~98% on the known classes. Full metrics are written to `results/metrics.md` by `scripts/run_pipeline.py`.